# Korean Olympiad problem clustering

This Colab notebook performs only the problem-type clustering stage:

1. Load the 150k `olympiads` rows from `AI-MO-NuminaMath-CoT-Ko`.
2. Generate Qwen3-Embedding-4B embeddings from the original English problems.
3. Compare candidate cluster counts using cosine silhouette, stability, and balance.
4. Fit the selected clustering to all rows and save labels, centroids, examples, and a clustered Parquet dataset.

Embedding generation is optimized for a 96 GB NVIDIA RTX PRO 6000 Blackwell GPU and checkpointed to Google Drive. Entropy and difficulty levels are intentionally deferred to a separate notebook.

In [ ]:
%pip install -q -U datasets huggingface_hub sentence-transformers accelerate pandas pyarrow scikit-learn tqdm

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
print("Using HF_TOKEN from Colab Secrets" if HF_TOKEN else "HF_TOKEN not found; using public access")

In [ ]:
# ---- User controls ----
DATASET_ID = "ChuGyouk/AI-MO-NuminaMath-CoT-Ko"
DATASET_SPLIT = "train"
SOURCE_FILTER = "olympiads"
DRIVE_ROOT = "/content/drive/MyDrive/Korean-TDCS/data/olympiad_clustering"
EXISTING_FILTERED_PARQUET = "/content/drive/MyDrive/Korean-TDCS/data/olympiad_dedup/numina_olympiads_train.parquet"

EMBEDDING_MODEL_ID = "Qwen/Qwen3-Embedding-4B"
EMBEDDING_DIMENSION = 1024
EMBEDDING_MAX_TOKENS = 1024
EMBEDDING_BATCH_SIZE = 512  # Increase if GPU utilization remains low; reduce only after CUDA OOM.
EMBEDDING_CHECKPOINT_ROWS = 8192
EMBEDDING_INSTRUCTION = (
    "Represent this Olympiad mathematics problem for clustering by mathematical topic, "
    "required concepts, and likely solution method."
)

CANDIDATE_CLUSTER_COUNTS = [10, 15, 20, 25, 30, 40]
CLUSTER_SELECTION_SAMPLE_SIZE = 30000
CLUSTER_MAX_ITERATIONS = 40
SELECTED_NUM_CLUSTERS = None  # None chooses automatically; set an integer to override.
RANDOM_SEED = 42
EXAMPLES_PER_CLUSTER = 3
DOWNLOAD_CLUSTERED_PARQUET = False

## 1. Paths, GPU validation, and helpers

In [ ]:
import gc
import hashlib
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Select a GPU runtime before continuing.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"GPU VRAM: {gpu.total_memory / 1024**3:.1f} GiB")
print(f"CUDA capability: {gpu.major}.{gpu.minor}")
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
os.environ["TOKENIZERS_PARALLELISM"] = "true"

drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(drive_root / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(drive_root / "hf_cache" / "datasets")

FILTERED_PARQUET = drive_root / "numina_olympiads_train.parquet"
CLUSTER_METRICS_PATH = drive_root / "cluster_count_metrics.csv"
CLUSTER_SIZES_PATH = drive_root / "cluster_sizes.csv"
CLUSTER_REPORT_PATH = drive_root / "cluster_examples.txt"
CLUSTERED_PARQUET_PATH = drive_root / "numina_olympiads_clustered.parquet"
CLUSTER_LABELS_PATH = drive_root / "cluster_labels.npy"
CLUSTER_CENTROIDS_PATH = drive_root / "cluster_centroids.npy"
CLUSTER_SUMMARY_PATH = drive_root / "clustering_summary.json"
DOWNLOAD_BUNDLE_PATH = drive_root / "olympiad_clustering_outputs.zip"


def dataset_fingerprint(values):
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value or "").encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()

## 2. Load or create the filtered 150k-row checkpoint

In [ ]:
from datasets import Dataset, load_dataset
from tqdm.auto import tqdm

existing_filtered = Path(EXISTING_FILTERED_PARQUET)
if FILTERED_PARQUET.exists():
    dataset_path = FILTERED_PARQUET
elif existing_filtered.exists():
    dataset_path = existing_filtered
else:
    print("No filtered checkpoint found; scanning the full dataset once...")
    stream = load_dataset(DATASET_ID, split=DATASET_SPLIT, streaming=True, token=HF_TOKEN)
    rows = []
    for row in tqdm(stream, desc="Scanning NuminaMath"):
        if row.get("source") == SOURCE_FILTER:
            rows.append(row)
    filtered = Dataset.from_list(rows)
    filtered.to_parquet(str(FILTERED_PARQUET))
    dataset_path = FILTERED_PARQUET
    print(f"Saved filtered checkpoint: {FILTERED_PARQUET}")

train_dataset = load_dataset("parquet", data_files=str(dataset_path), split="train")
required_columns = {"problem", "problem_ko", "solution_ko"}
if not required_columns.issubset(train_dataset.column_names):
    raise ValueError(f"Missing required columns: {required_columns - set(train_dataset.column_names)}")
if any(source != SOURCE_FILTER for source in set(train_dataset["source"])):
    raise ValueError("Filtered checkpoint contains a non-olympiads source")

embedding_texts = list(train_dataset["problem"])
fingerprint = dataset_fingerprint(embedding_texts)
print(f"Rows to cluster: {len(train_dataset):,}")
print(f"Dataset fingerprint: {fingerprint[:16]}...")

## 3. Generate resumable Qwen3-Embedding-4B embeddings

Each completed shard is saved immediately to Drive. Rerunning this cell skips valid shards. Embeddings are stored as float16 to reduce Drive usage and converted to float32 for clustering.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_config = {
    "dataset_fingerprint": fingerprint,
    "rows": len(train_dataset),
    "model_id": EMBEDDING_MODEL_ID,
    "dimension": EMBEDDING_DIMENSION,
    "max_tokens": EMBEDDING_MAX_TOKENS,
    "instruction": EMBEDDING_INSTRUCTION,
}
embedding_run_id = hashlib.sha256(
    json.dumps(embedding_config, sort_keys=True).encode("utf-8")
).hexdigest()[:12]
embedding_dir = drive_root / f"embeddings_{embedding_run_id}"
embedding_dir.mkdir(parents=True, exist_ok=True)
embedding_config_path = embedding_dir / "config.json"
embedding_config_path.write_text(json.dumps(embedding_config, indent=2), encoding="utf-8")
EMBEDDINGS_PATH = embedding_dir / "all_embeddings_float16.npy"

expected_parts = []
for start in range(0, len(embedding_texts), EMBEDDING_CHECKPOINT_ROWS):
    end = min(start + EMBEDDING_CHECKPOINT_ROWS, len(embedding_texts))
    expected_parts.append((start, end, embedding_dir / f"part_{start:06d}_{end:06d}.npy"))

def valid_embedding_part(path, expected_rows):
    if not path.exists():
        return False
    try:
        array = np.load(path, mmap_mode="r")
        return array.shape == (expected_rows, EMBEDDING_DIMENSION)
    except Exception:
        return False

missing_parts = [
    (start, end, path)
    for start, end, path in expected_parts
    if not valid_embedding_part(path, end - start)
]
print(f"Embedding shards complete: {len(expected_parts) - len(missing_parts)}/{len(expected_parts)}")

if missing_parts:
    embedding_model = SentenceTransformer(
        EMBEDDING_MODEL_ID,
        token=HF_TOKEN,
        device="cuda:0",
        model_kwargs={"torch_dtype": torch.bfloat16},
        truncate_dim=EMBEDDING_DIMENSION,
    )
    embedding_model.max_seq_length = EMBEDDING_MAX_TOKENS
    embedding_model.eval()
    print(f"Embedding explicitly on {embedding_model.device} with batch size {EMBEDDING_BATCH_SIZE:,}")

    for start, end, path in tqdm(missing_parts, desc="Embedding checkpoint shards"):
        shard = embedding_model.encode(
            embedding_texts[start:end],
            prompt=EMBEDDING_INSTRUCTION,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
            device="cuda:0",
        ).astype(np.float16)
        np.save(path, shard)
        print(f"Saved {path.name} ({end - start:,} rows)")

    del embedding_model
    gc.collect()
    torch.cuda.empty_cache()

if not EMBEDDINGS_PATH.exists():
    combined = np.lib.format.open_memmap(
        EMBEDDINGS_PATH, mode="w+", dtype=np.float16, shape=(len(train_dataset), EMBEDDING_DIMENSION)
    )
    for start, end, path in tqdm(expected_parts, desc="Combining embedding shards"):
        combined[start:end] = np.load(path)
    combined.flush()
    del combined

embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
if embeddings.shape != (len(train_dataset), EMBEDDING_DIMENSION):
    raise ValueError(f"Unexpected embedding shape: {embeddings.shape}")
print(f"Embeddings ready: {embeddings.shape}, stored at {EMBEDDINGS_PATH}")

## 4. GPU spherical k-means helpers

Because embeddings are normalized, spherical k-means groups problems by cosine similarity. Both cluster selection and final fitting run on the Blackwell GPU.

In [ ]:
from sklearn.metrics import adjusted_rand_score

@torch.inference_mode()
def spherical_kmeans(array, n_clusters, seed, max_iterations=40, chunk_size=32768):
    x = torch.as_tensor(np.asarray(array, dtype=np.float32), device="cuda:0")
    x = F.normalize(x, dim=1)
    generator = torch.Generator(device="cuda:0").manual_seed(seed)
    initial_indices = torch.randperm(len(x), generator=generator, device="cuda:0")[:n_clusters]
    centers = x[initial_indices].clone()

    for iteration in range(max_iterations):
        sums = torch.zeros((n_clusters, x.shape[1]), dtype=torch.float32, device="cuda:0")
        counts = torch.zeros(n_clusters, dtype=torch.long, device="cuda:0")
        labels_parts = []
        for start in range(0, len(x), chunk_size):
            chunk = x[start : start + chunk_size]
            labels = (chunk @ centers.T).argmax(dim=1)
            labels_parts.append(labels)
            sums.index_add_(0, labels, chunk)
            counts += torch.bincount(labels, minlength=n_clusters)

        empty = counts == 0
        if empty.any():
            replacements = torch.randperm(len(x), generator=generator, device="cuda:0")[: int(empty.sum())]
            sums[empty] = x[replacements]
            counts[empty] = 1
        new_centers = F.normalize(sums / counts.clamp_min(1).unsqueeze(1), dim=1)
        maximum_shift = (1 - (centers * new_centers).sum(dim=1)).max().item()
        centers = new_centers
        if maximum_shift < 1e-5:
            break

    final_labels = []
    for start in range(0, len(x), chunk_size):
        final_labels.append((x[start : start + chunk_size] @ centers.T).argmax(dim=1))
    labels = torch.cat(final_labels)
    return labels.cpu().numpy(), centers.cpu().numpy(), iteration + 1


@torch.inference_mode()
def cosine_silhouette_proxy(array, centers, chunk_size=32768):
    x = F.normalize(torch.as_tensor(np.asarray(array, dtype=np.float32), device="cuda:0"), dim=1)
    c = F.normalize(torch.as_tensor(centers, dtype=torch.float32, device="cuda:0"), dim=1)
    scores = []
    for start in range(0, len(x), chunk_size):
        top_two = torch.topk(x[start : start + chunk_size] @ c.T, k=2, dim=1).values
        within_distance = 1 - top_two[:, 0]
        next_distance = 1 - top_two[:, 1]
        silhouette = (next_distance - within_distance) / torch.maximum(
            torch.maximum(within_distance, next_distance), torch.tensor(1e-8, device="cuda:0")
        )
        scores.append(silhouette.cpu())
    return float(torch.cat(scores).mean())

## 5. Compare candidate cluster counts

The automatic selection score combines cosine separation, repeat-run stability, and cluster-size balance. The complete table is saved so `SELECTED_NUM_CLUSTERS` can be manually overridden after inspection.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
sample_size = min(CLUSTER_SELECTION_SAMPLE_SIZE, len(train_dataset))
sample_indices = rng.choice(len(train_dataset), size=sample_size, replace=False)
selection_embeddings = np.asarray(embeddings[sample_indices], dtype=np.float32)

metric_rows = []
for k in CANDIDATE_CLUSTER_COUNTS:
    print(f"Evaluating k={k}...")
    labels_a, centers_a, iterations_a = spherical_kmeans(
        selection_embeddings, k, RANDOM_SEED, CLUSTER_MAX_ITERATIONS
    )
    labels_b, _, _ = spherical_kmeans(
        selection_embeddings, k, RANDOM_SEED + 1, CLUSTER_MAX_ITERATIONS
    )
    silhouette = cosine_silhouette_proxy(selection_embeddings, centers_a)
    stability = float(adjusted_rand_score(labels_a, labels_b))
    counts = np.bincount(labels_a, minlength=k)
    proportions = counts / counts.sum()
    balance = float(-(proportions * np.log(proportions + 1e-12)).sum() / np.log(k))
    selection_score = silhouette + 0.05 * stability + 0.02 * balance
    metric_rows.append({
        "num_clusters": k,
        "cosine_silhouette_proxy": silhouette,
        "stability_ari": stability,
        "balance_score": balance,
        "minimum_cluster_rows_in_sample": int(counts.min()),
        "maximum_cluster_rows_in_sample": int(counts.max()),
        "iterations": iterations_a,
        "selection_score": selection_score,
    })

cluster_metrics = pd.DataFrame(metric_rows).sort_values("num_clusters")
cluster_metrics.to_csv(CLUSTER_METRICS_PATH, index=False)
display(cluster_metrics)

automatic_k = int(cluster_metrics.loc[cluster_metrics["selection_score"].idxmax(), "num_clusters"])
selected_k = int(SELECTED_NUM_CLUSTERS) if SELECTED_NUM_CLUSTERS is not None else automatic_k
if selected_k not in CANDIDATE_CLUSTER_COUNTS and SELECTED_NUM_CLUSTERS is None:
    raise ValueError("Automatic cluster count is invalid")
print(f"Automatically recommended clusters: {automatic_k}")
print(f"Cluster count used for final fit: {selected_k}")

## 6. Fit all 150k rows and save the clustered dataset

In [ ]:
all_embeddings_float32 = np.asarray(embeddings, dtype=np.float32)
cluster_labels, cluster_centroids, final_iterations = spherical_kmeans(
    all_embeddings_float32, selected_k, RANDOM_SEED, CLUSTER_MAX_ITERATIONS
)
np.save(CLUSTER_LABELS_PATH, cluster_labels.astype(np.int32))
np.save(CLUSTER_CENTROIDS_PATH, cluster_centroids.astype(np.float32))

clustered_dataset = train_dataset.add_column("cluster_id", cluster_labels.astype(int).tolist())
clustered_dataset.to_parquet(str(CLUSTERED_PARQUET_PATH))

cluster_sizes = pd.DataFrame({
    "cluster_id": np.arange(selected_k),
    "rows": np.bincount(cluster_labels, minlength=selected_k),
})
cluster_sizes["percentage"] = cluster_sizes["rows"] / len(train_dataset)
cluster_sizes.to_csv(CLUSTER_SIZES_PATH, index=False)
display(cluster_sizes)
print(f"Saved clustered dataset: {CLUSTERED_PARQUET_PATH}")

## 7. Create representative cluster examples

Examples nearest each centroid help determine what mathematical topic or solution style each cluster represents.

In [ ]:
report = []
for cluster_id in range(selected_k):
    indices = np.flatnonzero(cluster_labels == cluster_id)
    cluster_vectors = all_embeddings_float32[indices]
    similarities = cluster_vectors @ cluster_centroids[cluster_id]
    nearest_local = np.argsort(-similarities)[:EXAMPLES_PER_CLUSTER]
    representative_indices = indices[nearest_local]

    report.append("=" * 100)
    report.append(f"CLUSTER {cluster_id} | ROWS: {len(indices):,} | SHARE: {len(indices) / len(train_dataset):.2%}")
    for rank, dataset_index in enumerate(representative_indices, start=1):
        row = train_dataset[int(dataset_index)]
        report.append(f"--- REPRESENTATIVE {rank} | DATASET INDEX {dataset_index} ---")
        report.append(f"ENGLISH PROBLEM:\n{row['problem']}")
        report.append(f"KOREAN PROBLEM:\n{row['problem_ko']}")

report_text = "\n".join(report)
CLUSTER_REPORT_PATH.write_text(report_text, encoding="utf-8")
print(report_text)
print(f"\nSaved representative examples: {CLUSTER_REPORT_PATH}")

## 8. Save summary and download outputs

All artifacts remain in Drive. The compact bundle downloads automatically; enable `DOWNLOAD_CLUSTERED_PARQUET` if you also want the large Parquet file downloaded directly.

In [ ]:
import zipfile
from google.colab import files

summary = {
    "dataset_id": DATASET_ID,
    "source_filter": SOURCE_FILTER,
    "rows": len(train_dataset),
    "embedding_model": EMBEDDING_MODEL_ID,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "embedding_max_tokens": EMBEDDING_MAX_TOKENS,
    "embedding_run_id": embedding_run_id,
    "automatic_cluster_count": automatic_k,
    "selected_cluster_count": selected_k,
    "final_kmeans_iterations": final_iterations,
    "clustered_parquet": str(CLUSTERED_PARQUET_PATH),
    "embeddings": str(EMBEDDINGS_PATH),
    "cluster_labels": str(CLUSTER_LABELS_PATH),
    "cluster_centroids": str(CLUSTER_CENTROIDS_PATH),
}
CLUSTER_SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))

with zipfile.ZipFile(DOWNLOAD_BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [
        CLUSTER_SUMMARY_PATH, CLUSTER_METRICS_PATH, CLUSTER_SIZES_PATH,
        CLUSTER_REPORT_PATH, CLUSTER_LABELS_PATH, CLUSTER_CENTROIDS_PATH,
    ]:
        archive.write(path, arcname=path.name)
files.download(str(DOWNLOAD_BUNDLE_PATH))

if DOWNLOAD_CLUSTERED_PARQUET:
    files.download(str(CLUSTERED_PARQUET_PATH))
else:
    print(f"Large clustered dataset remains saved at: {CLUSTERED_PARQUET_PATH}")